#### Heart disease sınıflandıracaz 

In [ ]:
import numpy as np
import pandas as pd

sutunlar = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "target",
]

#### Temizleyek

In [3]:
df = pd.read_csv("processed.cleveland.data", names=sutunlar, na_values="?")
df = df.dropna()
df

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,57.0,0.0,4.0,140.0,241.0,0.0,0.0,123.0,1.0,0.2,2.0,0.0,7.0,1
298,45.0,1.0,1.0,110.0,264.0,0.0,0.0,132.0,0.0,1.2,2.0,0.0,7.0,1
299,68.0,1.0,4.0,144.0,193.0,1.0,0.0,141.0,0.0,3.4,2.0,2.0,7.0,2
300,57.0,1.0,4.0,130.0,131.0,0.0,0.0,115.0,1.0,1.2,2.0,1.0,7.0,3


#### normalde 4 sınıf var biz binary hale getiriyoruz çünkü neden olmasın : 

#### 1 - Hasta, 
#### 0 - Sağlıklı

In [6]:
yc = (df["target"] > 0).astype(int)
Xc = df.drop("target", axis=1)

print("Satır sayısı:", Xc.shape[0])

Satır sayısı: 297


#### eğitim ve testi ayıralım

In [7]:
from sklearn.model_selection import train_test_split

SEED = 42
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.2, random_state=SEED, stratify=yc
)

print(f"Eğitim hasta sayısı: {Xc_train.shape[0]}")
print(f"Test hasta sayısı: {Xc_test.shape[0]}")

Eğitim hasta sayısı: 237
Test hasta sayısı: 60


#### Sınıflandırıcıları yarıştıralım

In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [10]:
yarismacilarimiz_yasarlar = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic": make_pipeline(StandardScaler(), LogisticRegression()),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
    "Karar ağacı": DecisionTreeClassifier(random_state=SEED),
    "Rastgele orman": RandomForestClassifier(random_state=SEED),
}

satirlar = []

for ad, m in yarismacilarimiz_yasarlar.items():
  m.fit(Xc_train, yc_train)
  satirlar.append({
      "Model": ad,
      "Eğitim": accuracy_score(yc_train, m.predict(Xc_train)), 
      #model ezberlemiş mi alttakiyle ilişkisine bakacaz: 
      # train'de predict 100, testte 60 ise ezberlemiş mesela
      "Test": accuracy_score(yc_test, m.predict(Xc_test)),
  })
  
tablo = pd.DataFrame(satirlar).set_index("Model").round(3)
display(tablo)

,Eğitim,Test
Model,,
Baseline,0.540,0.533
Logistic,0.852,0.833
KNN,0.869,0.883
Karar ağacı,1.000,0.700
Rastgele orman,1.000,0.850


#### Cross-validation yapalım

In [12]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_satirlar = []

for ad, m in yarismacilarimiz_yasarlar.items():
  skorlar = cross_val_score(m, Xc_train, yc_train, cv=cv, scoring="accuracy")
  cv_satirlar.append({
      "Model": ad,
      "Ortalama skor": round(skorlar.mean(), 3),
      "Standart sapma": round(skorlar.std(), 3),
  })

cv_sonuc = (
    pd.DataFrame(cv_satirlar)
    .set_index("Model")
    .sort_values("Ortalama skor", ascending=False)
)
display(cv_sonuc)

,Ortalama skor,Standart sapma
Model,,
Logistic,0.827,0.059
Rastgele orman,0.789,0.079
KNN,0.776,0.076
Karar ağacı,0.721,0.071
Baseline,0.540,0.008


#### KNN İçin Hiperparametre Optimizasyonu (GridSearchCV)

In [13]:
from sklearn.model_selection import GridSearchCV

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())

param_grid = {
    "kneighborsclassifier__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21],
    "kneighborsclassifier__weights": ["uniform", "distance"],
}

opt = GridSearchCV(pipe, param_grid, cv=cv, scoring="accuracy")
opt.fit(Xc_train, yc_train)

print("En iyi parametre:", opt.best_params_)
print(f"En iyi CV skoru: {opt.best_score_:.3f}")

En iyi parametre: {'kneighborsclassifier__n_neighbors': 21, 'kneighborsclassifier__weights': 'uniform'}
En iyi CV skoru: 0.827
